# Automatické tréningy pre rôzne n_input a horizonty (DST+1 … DST+6)

Tento notebook načíta dáta **iba raz** a potom postupne vytvorí a natrénuje samostatný model pre každú kombináciu:
- `n_input` ∈ {6, 12, 18, 24, 30, 36, 42, 48}
- `y_col` = `DST+1` … `DST+6`

Model/weighty sa ukladajú do `models/` a súhrn metrík do `results/summary.csv`.


In [1]:
import os
import numpy as np
import pandas as pd

from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from keras.models import Model
from keras.layers import Dense, Input, LSTM, Flatten, TimeDistributed, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Reprodukovateľnosť (voliteľné)
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)


2026-04-10 14:14:37.204145: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os; 
print(os.getcwd())

/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/3_modelovanie/train automate


In [3]:
# =========================
# Nastavenia
# =========================

# Cesty k datasetom (ponechané ako v pôvodnom notebooku)
BASE_PATH = "/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/0_datasety/"
file_path_train = BASE_PATH + "train_omni.csv"
file_path_test  = BASE_PATH + "test_omni.csv"


# Tréningové nastavenia
BATCH_SIZE = 256
EPOCHS = 200
PATIENCE = 25  # early stopping na val_mae

# Kombinácie na tréning
#N_INPUT_LIST = [36, 42, 48]
N_INPUT_LIST = [6, 12, 18, 24, 30, 36, 42, 48]
HORIZONS = [3]

# Výstupné priečinky
MODELS_DIR = "models"
RESULTS_DIR = "results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [4]:
# =========================
# Načítanie dát (iba raz)
# =========================
train_raw = pd.read_csv(file_path_train)
test_raw  = pd.read_csv(file_path_test)

# Skontroluj, aké DST+ stĺpce existujú (pomôže odhaliť preklepy v názvoch)
dst_cols = [c for c in train_raw.columns if c.startswith("DST+")]
print("DST+ stĺpce v train:", dst_cols)
print("DST+ stĺpce v test :", [c for c in test_raw.columns if c.startswith("DST+")])

# čas
if "time1" in train_raw.columns:
    train_raw["time1"] = pd.to_datetime(train_raw["time1"])
if "time1" in test_raw.columns:
    test_raw["time1"] = pd.to_datetime(test_raw["time1"])


DST+ stĺpce v train: ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']
DST+ stĺpce v test : ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']


In [5]:
def build_model(n_input: int, n_features: int) -> keras.Model:
    """LSTM model pre multivariačný vstup."""
    inputs = Input(shape=(n_input, n_features))

    x = Bidirectional(
        LSTM(128, return_sequences=True, dropout=0.1, recurrent_dropout=0.1)
    )(inputs)
    x = LSTM(128, return_sequences=True)(x)
    x = TimeDistributed(Dense(1, activation="linear"))(x)
    x = Flatten()(x)
    outputs = Dense(1, activation="linear")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss="mse", optimizer="adam", metrics=["mae"])
    return model


def make_splits(train_df: pd.DataFrame, test_df: pd.DataFrame, y_col: str, predictors=None):
    """Pripraví train/valid/test dáta pre daný y_col (bez NaN)."""

    if predictors is None:
        predictors = ["DST", "v"]   

    features = predictors + [y_col]

    train = train_df[features].copy()
    test = test_df[features].copy()

    # odstránenie NaN
    train = train.dropna().reset_index(drop=True)
    test = test.dropna().reset_index(drop=True)

    # časový split train/valid
    valid_size = int(len(train) * 0.2)

    if valid_size == 0:
        raise ValueError(f"Príliš málo dát po dropna pre {y_col}")

    valid = train.iloc[-valid_size:, :].copy()
    train = train.iloc[:-valid_size, :].copy()

    # X = 2 features
    X_train = train[predictors].values
    y_train = train[y_col].values

    X_val = valid[predictors].values
    y_val = valid[y_col].values

    X_test = test[predictors].values
    y_test = test[y_col].values

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), predictors

In [6]:
def train_one(y_col: str, n_input: int):
    predictors = ["DST", "v"]   

    (X_train, y_train), (X_val, y_val), (X_test, y_test), predictors = make_splits(
        train_raw, test_raw, y_col, predictors=predictors
    )

    train_gen = TimeseriesGenerator(X_train, y_train, length=n_input, batch_size=BATCH_SIZE)
    val_gen   = TimeseriesGenerator(X_val, y_val, length=n_input, batch_size=BATCH_SIZE)
    test_gen  = TimeseriesGenerator(X_test, y_test, length=n_input, batch_size=BATCH_SIZE)

    if len(train_gen) == 0 or len(val_gen) == 0:
        print(f"[SKIP] {y_col}, n_input={n_input} – prázdny generátor")
        return None

    n_features = len(predictors)
    model = build_model(n_input, n_features)

    print("Predictors:", predictors)
    print("X_train shape:", X_train.shape)
    print("First batch X shape:", train_gen[0][0].shape)
    print("Model input shape:", model.input_shape)

    model_path = os.path.join(MODELS_DIR, f"{y_col}_{n_input}H_V.keras")
    checkpoint = ModelCheckpoint(model_path, monitor="val_mae", verbose=1, save_best_only=True, mode="min")
    early = EarlyStopping(monitor="val_mae", mode="min", patience=PATIENCE, restore_best_weights=True)
    callbacks = [checkpoint, early]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        verbose=1,
        callbacks=callbacks
    )

    test_loss, test_mae = model.evaluate(test_gen, verbose=0)

    hist_df = pd.DataFrame(history.history)
    hist_csv = os.path.join(RESULTS_DIR, f"history_{y_col}_{n_input}H_V.csv")
    hist_df.to_csv(hist_csv, index=False)

    return {
        "y_col": y_col,
        "horizon_hours": int(y_col.split("+")[1]),
        "n_input": n_input,
        "best_val_mae": float(np.min(hist_df["val_mae"])) if "val_mae" in hist_df else np.nan,
        "best_val_loss": float(np.min(hist_df["val_loss"])) if "val_loss" in hist_df else np.nan,
        "test_mae": float(test_mae),
        "test_loss": float(test_loss),
        "model_path": model_path,
        "history_path": hist_csv,
        "epochs_ran": int(len(hist_df)),
    }

In [7]:
# =========================
# Spustenie všetkých tréningov
# =========================
summary_rows = []

for h in HORIZONS:
    y_col = f"DST+{h}"

    # ochrana, ak by stĺpec neexistoval
    if y_col not in train_raw.columns or y_col not in test_raw.columns:
        print(f"[SKIP] {y_col} neexistuje v datasete.")
        continue

    for n_input in N_INPUT_LIST:
        print("\n" + "="*80)
        print(f"Trénujem: y_col={y_col}, n_input={n_input}")
        print("="*80)

        row = train_one(y_col, n_input)
        if row is not None:
            summary_rows.append(row)


summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(RESULTS_DIR, "summary3_V.csv")
summary.to_csv(summary_path, index=False)

summary.sort_values(["horizon_hours", "n_input"]).head(20), summary_path



Trénujem: y_col=DST+3, n_input=6
Predictors: ['DST', 'v']
X_train shape: (229356, 2)
First batch X shape: (256, 6, 2)
Model input shape: (None, 6, 2)
Epoch 1/200


/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 324.1630 - mae: 10.8126
Epoch 1: val_mae improved from inf to 8.48264, saving model to models/DST+3_6H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 91s 90ms/step - loss: 324.0381 - mae: 10.8101 - val_loss: 253.3985 - val_mae: 8.4826
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 136.5532 - mae: 7.0353
Epoch 2: val_mae improved from 8.48264 to 8.02143, saving model to models/DST+3_6H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 76s 85ms/step - loss: 136.5302 - mae: 7.0349 - val_loss: 200.3534 - val_mae: 8.0214
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 104.7256 - mae: 6.5554
Epoch 3: val_mae improved from 8.02143 to 7.20158, saving model to models/DST+3_6H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 77s 86ms/step - loss: 104.7246 - mae: 6.5553 - val_loss: 161.3073 - val_mae: 7.2016
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 99.5411 - mae: 6.3461
Epoch 4: val_mae did not improve from 7.20158
896/896 ━━━━━━━━━

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 295.8109 - mae: 10.4678
Epoch 1: val_mae improved from inf to 8.32137, saving model to models/DST+3_12H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 147s 153ms/step - loss: 295.7079 - mae: 10.4656 - val_loss: 241.6099 - val_mae: 8.3214
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - loss: 140.5595 - mae: 7.4076
Epoch 2: val_mae did not improve from 8.32137
896/896 ━━━━━━━━━━━━━━━━━━━━ 142s 159ms/step - loss: 140.5489 - mae: 7.4073 - val_loss: 249.8677 - val_mae: 10.3674
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - loss: 129.5953 - mae: 7.0422
Epoch 3: val_mae did not improve from 8.32137
896/896 ━━━━━━━━━━━━━━━━━━━━ 138s 154ms/step - loss: 129.5777 - mae: 7.0419 - val_loss: 266.5971 - val_mae: 9.3667
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 112.4558 - mae: 6.7339
Epoch 4: val_mae improved from 8.32137 to 7.70628, saving model to models/DST+3_12H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 141s 157ms/step - l

...

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - loss: 374.4382 - mae: 12.6394
Epoch 1: val_mae improved from inf to 9.65929, saving model to models/DST+3_42H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 496s 540ms/step - loss: 374.3321 - mae: 12.6373 - val_loss: 281.6269 - val_mae: 9.6593
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - loss: 183.8463 - mae: 8.5088
Epoch 2: val_mae improved from 9.65929 to 7.93226, saving model to models/DST+3_42H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 482s 538ms/step - loss: 183.8288 - mae: 8.5083 - val_loss: 195.8310 - val_mae: 7.9323
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step - loss: 137.1151 - mae: 7.4392
Epoch 3: val_mae did not improve from 7.93226
896/896 ━━━━━━━━━━━━━━━━━━━━ 518s 578ms/step - loss: 137.1232 - mae: 7.4394 - val_loss: 181.0369 - val_mae: 8.2350
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step - loss: 132.6811 - mae: 7.4072
Epoch 4: val_mae improved from 7.93226 to 7.71946, saving model to models/DST+3_42H_V.keras
896

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step - loss: 430.8360 - mae: 13.9319
Epoch 1: val_mae improved from inf to 13.62866, saving model to models/DST+3_48H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 338s 364ms/step - loss: 430.7592 - mae: 13.9304 - val_loss: 490.0994 - val_mae: 13.6287
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step - loss: 203.2115 - mae: 9.3202
Epoch 2: val_mae improved from 13.62866 to 10.81006, saving model to models/DST+3_48H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 323s 360ms/step - loss: 203.2218 - mae: 9.3202 - val_loss: 333.4841 - val_mae: 10.8101
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step - loss: 173.3166 - mae: 8.3960
Epoch 3: val_mae improved from 10.81006 to 8.98506, saving model to models/DST+3_48H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 323s 360ms/step - loss: 173.3140 - mae: 8.3959 - val_loss: 231.7900 - val_mae: 8.9851
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 709ms/step - loss: 145.7060 - mae: 7.7703
Epoch 4: val_mae did not improve from 8.985

(   y_col  horizon_hours  n_input  best_val_mae  best_val_loss  test_mae  \
 0  DST+3              3        6      6.824423     120.693138  5.017393   
 1  DST+3              3       12      6.860461     127.793503  5.014315   
 2  DST+3              3       18      6.781991     122.581169  4.967137   
 3  DST+3              3       24      6.905131     132.224289  4.975629   
 4  DST+3              3       30      6.778080     118.542213  4.921163   
 5  DST+3              3       36      6.858864     128.491226  5.091484   
 6  DST+3              3       42      6.784397     124.617149  4.916238   
 7  DST+3              3       48      6.748437     124.713402  4.962082   
 
    test_loss                model_path                     history_path  \
 0  60.388241   models/DST+3_6H_V.keras   results/history_DST+3_6H_V.csv   
 1  61.552856  models/DST+3_12H_V.keras  results/history_DST+3_12H_V.csv   
 2  60.627857  models/DST+3_18H_V.keras  results/history_DST+3_18H_V.csv   
 3  62.843

## Poznámky
- Ak chceš presne poradie ako si písala (najprv `DST+1` pre všetky `n_input`, potom `DST+2`, …), tak to presne robí horný loop (horizonty vonkajší, `n_input` vnútorný).
- Ak chceš opačne (pre dané `n_input` spraviť `DST+1..6`), stačí prehodiť poradie cyklov.
